# Week 2 Day 2 Challenge - a word lookup, a memory about each customer, and a research sub-agent

The lab gave the UnderCover agent a long-term memory: the facts in `data/knowledge.json`, kept in Qdrant and searched by meaning over MCP. This notebook takes the lab's assignment in three steps and runs each as an experiment in Langfuse:

1. **A word lookup in place of Qdrant.** A tool that finds the facts sharing words with its query. Does it find the facts the claims need as well as searching by meaning, and does that change the decisions?
2. **A memory about each customer.** A Markdown file of notes for each `customer_id`, with a tool to read them and a tool to add to them. What should happen to the notes between runs?
3. **Stretch: a research sub-agent.** The retrieval moves into a sub-agent with `qdrant-find` and the customer notes, and the UnderCover agent calls it as a tool.

The claims, the instructions, the tools and the scores are the ones from `labs/week2day2.ipynb`. The agent, the experiment and the score code are the lab's, with the changes named where they come.

## Why Claude Haiku 5.5 and not GPT-6 Luna

As in my Week 2 Day 1 challenge: the lab calls GPT-6 Luna through OpenRouter, my OpenRouter account has no credit and I can't buy any, so every agent here runs on Claude Haiku 5.5 through Anthropic's API, at `reasoning_effort="low"` like the lab's Luna. Haiku 5.5 costs the same as Luna ($0.10 per million input tokens and $0.50 per million output tokens). The cost of the model calls is the tokens LangChain reports times those prices, since only OpenRouter sends a cost with each call.

All the experiments run on Haiku, so they're compared with each other, not with the lab's runs. I ran the lab's free version (`labs/week2day2free.ipynb`) unchanged on 5 claims; its numbers are in my fork.

What you need: `ANTHROPIC_API_KEY` and the Langfuse keys in `.env`, and the packages from the lab plus `langchain-anthropic`.

In [ ]:
import json
import os
import re
import shutil
import sys
import time
from contextvars import ContextVar
from functools import cached_property
from pathlib import Path
from typing import Literal

import anthropic
import httpx2
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langfuse import Evaluation, get_client
from langfuse.langchain import CallbackHandler

# This notebook sits two folders below the repo's root, and the payout rules are in labs/
ROOT = Path("../..")
sys.path.append(str(ROOT / "labs"))
from week2day1_payout import Benefit, Line, Payout, work_out_payout  # noqa: E402

## Connect to Anthropic and Langfuse

The keys come from the `.env` file in the `undercover` folder.

In [ ]:
load_dotenv(ROOT / ".env", override=True)

if os.getenv("ANTHROPIC_API_KEY"):
    print("Anthropic API key found")
else:
    print("No Anthropic API key found. Add ANTHROPIC_API_KEY to the .env file")

langfuse = get_client()

if langfuse.auth_check():
    print("Connected to Langfuse")
else:
    print("No Langfuse keys found. Add them to the .env file")

## The even harder claims and the knowledge base

The lab's 25 claims, with the handlers' decisions and the facts each one needs, and the knowledge base the handlers used.

In [ ]:
EVEN_HARDER = ROOT / "data" / "evals" / "even_harder"


def load(name):
    """Read one of the JSON files of the even harder claims."""
    return json.loads((EVEN_HARDER / name).read_text(encoding="utf-8"))


claims = load("claims.json")
labels = {decision["claim_id"]: decision for decision in load("decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (ROOT / "data" / "policy_wording.md").read_text(encoding="utf-8")
knowledge = json.loads((ROOT / "data" / "knowledge.json").read_text(encoding="utf-8"))

needing_a_fact = sum(1 for label in labels.values() if label["needs"])
print(f"{len(claims)} claims, {needing_a_fact} of them need a fact from the knowledge base, which has {len(knowledge)} facts")

## The memory over MCP

The lab's Windows adjustment, then Qdrant's MCP server with an empty memory. As in the lab, `--python 3.12.12` makes `uvx` run the server on a Python it works with, a workaround for a known issue with the server. The free version of the lab stores each fact with `qdrant-store` itself instead of asking the agent to, and so does this notebook: every experiment then searches the same memory, and it costs nothing.

In [ ]:
if sys.platform == "win32":
    import subprocess
    from functools import partial

    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

In [ ]:
MEMORY = Path("qdrant_data").resolve()
shutil.rmtree(MEMORY, ignore_errors=True)

MEMORY_ENV = {"QDRANT_LOCAL_PATH": str(MEMORY), "COLLECTION_NAME": "memory"}
client = MultiServerMCPClient({"memory": {"transport": "stdio", "command": "uvx", "args": ["--python", "3.12.12", "mcp-server-qdrant@0.8.1"], "env": MEMORY_ENV}})
memory_tools = {memory_tool.name: memory_tool for memory_tool in await client.get_tools()}

for entry in knowledge:
    await memory_tools["qdrant-store"].ainvoke({"information": entry["fact"]})
print(f"Stored {len(knowledge)} facts")

## The UnderCover agent

The lab's instructions, prompt, `record_decision` and payout tool, unchanged.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
"""

PAYOUT_TOOL_LINE = "Use the payout tool to work out the payout on an approved claim, and record the payout it returns.\n"
MEMORY_LINE = "Before you decide, search your memory for any product, event or place in the claim that you need to know more about.\n"


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

In [ ]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"


@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

### The model

Langfuse runs each experiment on its own event loop, and an HTTP connection opened on one loop can't be used on another. The lab gives Luna an HTTP client that keeps no connections open between calls. `ChatAnthropic` has no setting for that: it shares one async client between every model in the process, and in a test the first call on a second event loop failed with a connection error. So this subclass gives Haiku its own client, which keeps no connections open. The Anthropic SDK uses its own HTTP package, `httpx2`, so the limits come from there.

In [ ]:
MODEL = "claude-haiku-5-5"
INPUT_PRICE = 0.10
OUTPUT_PRICE = 0.50


class ChatAnthropicFreshConnections(ChatAnthropic):
    """ChatAnthropic with an async HTTP client that keeps no connections open between calls."""

    @cached_property
    def _async_client(self) -> anthropic.AsyncClient:
        http_client = anthropic.DefaultAsyncHttpxClient(limits=httpx2.Limits(max_keepalive_connections=0))
        return anthropic.AsyncClient(**self._client_params, http_client=http_client)


haiku = ChatAnthropicFreshConnections(model=MODEL, reasoning_effort="low", max_tokens=16000, timeout=120, max_retries=5)

## Challenge 1: a word lookup

`search_facts` splits its query into words, leaves out common words like "the" and "of", and returns the facts that share the most words with it, up to 10 like `qdrant-find`. There is no stemming, so "storm" doesn't find "storms". It returns the facts word for word, so the lab's `found_fact` score works on it unchanged.

In [ ]:
STOP_WORDS = set("a an and are as at be by for from has have how in is it its of on or that the this to was were what when where which who with".split())


def words(text):
    """The words in a text, in lower case, without the common ones."""
    return set(re.findall(r"[a-z0-9]+", text.lower())) - STOP_WORDS


FACT_WORDS = [(entry["fact"], words(entry["fact"])) for entry in knowledge]


@tool(parse_docstring=True)
def search_facts(query: str) -> str:
    """Search the knowledge base for facts that share words with the query: products, storms, strikes, dive sites, rivers and activity operators.

    Args:
        query: The words to look for, such as a product's name or a storm's name
    """
    query_words = words(query)
    scored = sorted(((len(query_words & fact_words), fact) for fact, fact_words in FACT_WORDS), key=lambda pair: -pair[0])
    found = [fact for score, fact in scored if score > 0][:10]
    return "\n".join(found) if found else "No facts share a word with the query"

The lab's query, `a gadget for watching films on a wall`, shares no words with the projector's fact, and a query that names the thing finds it either way. Here are the top 3 from each search.

In [ ]:
QUERIES = ["a gadget for watching films on a wall", "Halden Arc", "Hurricane Marisol Bahamas flights"]

for query in QUERIES:
    by_meaning = json.loads((await memory_tools["qdrant-find"].ainvoke({"query": query}))[0]["text"])[1:4]
    by_words = search_facts.invoke({"query": query}).split("\n")[:3]
    print(f"Query: {query}")
    for name, results in [("Qdrant", by_meaning), ("Words", by_words)]:
        for result in results:
            print(f"  {name}: {re.sub('<[^>]+>', ' ', result).strip()[:110]}")
    print()

### The agents

The lab's two agents and a third with the word lookup in place of `qdrant-find`. Each one is told to search its memory in the same words, so only the search changes. The Qdrant server runs read-only, as in the lab.

In [ ]:
read_only_client = MultiServerMCPClient({
    "memory": {"transport": "stdio", "command": "uvx", "args": ["--python", "3.12.12", "mcp-server-qdrant@0.8.1"], "env": {**MEMORY_ENV, "QDRANT_READ_ONLY": "true"}}
})
qdrant_find = [memory_tool for memory_tool in await read_only_client.get_tools() if memory_tool.name == "qdrant-find"][0]

SYSTEM_PROMPT = f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}\n{policy_wording}\n"
MEMORY_PROMPT = f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{MEMORY_LINE}\n{policy_wording}\n"

payout_agent = create_agent(model=haiku, tools=[record_decision, calculate_payout], system_prompt=SYSTEM_PROMPT)
qdrant_agent = create_agent(model=haiku, tools=[record_decision, calculate_payout, qdrant_find], system_prompt=MEMORY_PROMPT, middleware=[ToolRetryMiddleware(max_retries=3)])
words_agent = create_agent(model=haiku, tools=[record_decision, calculate_payout, search_facts], system_prompt=MEMORY_PROMPT)

## The scores

The lab's scores, with the cost of the model calls, the time and the number of searches a claim kept beside them.

In [ ]:
DECISIONS = ["approve", "decline", "refer"]
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer, expected):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer, expected):
    """What the agent's answer costs UnderCover, against the handler's decision."""
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
    if answer["decision"] == "decline":
        return WRONGFUL_DECLINE
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL


def found_fact(answer, expected):
    """For a claim that needs a fact from the knowledge base, whether the agent's searches found it."""
    if not expected["needs"]:
        return None
    return all(fact in answer["retrieved"] for fact in expected["needs"])


def kappa(table):
    """Cohen's kappa for the agent's decisions against the handlers'."""
    observed = (table["agent"] == table["handler"]).mean()
    agent_share = table["agent"].value_counts(normalize=True)
    handler_share = table["handler"].value_counts(normalize=True)
    expected = sum(agent_share.get(decision, 0) * handler_share.get(decision, 0) for decision in DECISIONS)
    return (observed - expected) / (1 - expected)


def show_confusion_matrix(table, title):
    """Draw the confusion matrix, with the handler's decision on each row and the agent's in each column."""
    matrix = pd.crosstab(table["handler"], table["agent"]).reindex(index=DECISIONS, columns=DECISIONS, fill_value=0)
    fig = px.imshow(matrix, text_auto=True, color_continuous_scale="Blues", labels={"x": "Agent", "y": "Handler", "color": "Claims"}, title=title)
    fig.show()

In [ ]:
def score_claim(*, output, expected_output, **kwargs):
    """Item evaluator: whether the agent got the claim right, the cost of its mistake, and whether it found the fact it needed."""
    evaluations = [
        Evaluation(name="right", value=int(is_right(output, expected_output))),
        Evaluation(name="mistake_cost", value=mistake_cost(output, expected_output)),
    ]
    found = found_fact(output, expected_output)
    if found is not None:
        evaluations.append(Evaluation(name="found_fact", value=int(found)))
    return evaluations


def score_table(item_results):
    """One row for each claim in a run: the agent's answer beside the handler's, with the scores."""
    rows = []
    for result in item_results:
        answer, expected = result.output, result.item.expected_output
        rows.append({
            "claim_id": result.item.id,
            "handler": expected["decision"],
            "agent": answer["decision"],
            "handler_payout": expected["payout"],
            "agent_payout": answer["payout"],
            "right": is_right(answer, expected),
            "mistake_cost": mistake_cost(answer, expected),
            "found_fact": found_fact(answer, expected),
            "searches": answer["searches"],
            "llm_cost": answer["cost"],
            "seconds": answer["seconds"],
            "rationale": answer["rationale"],
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def run_scores(table):
    """The scores for a whole run."""
    agent_approvals = table[table["agent"] == "approve"]
    handler_approvals = table[table["handler"] == "approve"]
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "precision": (agent_approvals["handler"] == "approve").mean(),
        "recall": (handler_approvals["agent"] == "approve").mean(),
        "kappa": kappa(table),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "straight_through": (table["agent"] != "refer").mean(),
        "found_fact": table["found_fact"].dropna().astype(float).mean(),
        "searches_a_claim": table["searches"].mean(),
        "llm_cost": table["llm_cost"].sum(),
        "seconds_a_claim": table["seconds"].mean(),
    }


def score_run(*, item_results, **kwargs):
    """Run evaluator: the scores for the whole run."""
    scores = run_scores(score_table(item_results))
    return [Evaluation(name=name, value=float(value)) for name, value in scores.items()]


def average_scores(table):
    """The scores for each run of an experiment, averaged over the runs."""
    return pd.DataFrame([run_scores(run_table) for _, run_table in table.groupby("run")]).mean()

## The dataset

The lab's `even-harder-claims` dataset. If you ran the lab it's already in Langfuse, and this updates its items in place.

`TINY = True` runs 5 of the claims. `N_TIMES` is the number of runs of each experiment: I use 2, so a difference has to show up twice. The experiments that search Qdrant run one claim at a time, because Qdrant's local mode lets only one process open the database, and the others run 5 at once.

In [ ]:
DATASET = "even-harder-claims"

langfuse.create_dataset(name=DATASET, description="UnderCover's even harder claims with the handlers' decisions")

for claim in claims:
    label = labels[claim["claim_id"]]
    langfuse.create_dataset_item(
        dataset_name=DATASET,
        id=claim["claim_id"],
        input=claim,
        expected_output={"decision": label["decision"], "payout": label["payout"], "reason": label["reason"], "needs": label["needs"]},
        metadata={"tiny": label["tiny"]},
    )

In [ ]:
TINY = False
N_TIMES = 2

dataset = langfuse.get_dataset(DATASET)
items = dataset.items
if TINY:
    items = [item for item in items if item.metadata["tiny"]]

print(f"Running {len(items)} of the {len(dataset.items)} claims, {N_TIMES} times for each experiment")

## The task

`decide` is the lab's, with three changes:

- **The search tools are a set.** The lab picks out the search results by the name `qdrant-find`; here any tool in `SEARCH_TOOLS` counts, so the word lookup is scored the same way.
- **A sub-agent's messages count too.** The research sub-agent in the stretch challenge searches inside its own conversation, which the UnderCover agent never sees. It hands its messages back through `SUB_AGENT_MESSAGES`, a context variable that `decide` sets for each claim, so its searches and its model calls are counted with the claim's.
- **A failed claim counts as a referral.** If the agent stops with an error, or without recording a decision, the claim would go to a handler. `decide` streams the agent's steps instead of waiting for the end, so a claim that fails part way still counts the model calls and searches it made.

In [ ]:
SEARCH_TOOLS = {"qdrant-find", "search_facts"}
SUB_AGENT_MESSAGES = ContextVar("sub_agent_messages")


async def decide(agent, claim):
    """Run an agent on one claim and return its decision, the cost and time of its model calls and what its searches returned."""
    started = time.perf_counter()
    sub_agent_messages = []
    token = SUB_AGENT_MESSAGES.set(sub_agent_messages)
    messages, failed = [], ""
    try:
        async for state in agent.astream(
            {"messages": [{"role": "user", "content": make_user_prompt(claim)}]},
            config={"callbacks": [CallbackHandler()]},
            stream_mode="values",
        ):
            messages = state["messages"]
    except Exception as error:
        failed = f"Failed: {type(error).__name__}: {error}"
    finally:
        SUB_AGENT_MESSAGES.reset(token)
    messages = messages + sub_agent_messages
    seconds = time.perf_counter() - started

    replies = [message for message in messages if message.type == "ai"]
    input_tokens = sum((message.usage_metadata or {}).get("input_tokens", 0) for message in replies)
    output_tokens = sum((message.usage_metadata or {}).get("output_tokens", 0) for message in replies)
    retrieved = "\n".join(message.text for message in messages if message.type == "tool" and message.name in SEARCH_TOOLS)
    searches = sum(1 for message in replies for call in message.tool_calls if call["name"] in SEARCH_TOOLS)

    answer = recorded.get(claim["claim_id"], {"decision": "refer", "payout": 0.0, "rationale": failed or "Nothing recorded"})
    print(f"{claim['claim_id']}: {answer['decision']} ${answer['payout']:,.2f}{'  ' + failed[:100] if failed else ''}")
    return {
        **answer,
        "cost": (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000,
        "seconds": seconds,
        "retrieved": retrieved,
        "searches": searches,
    }

## The experiments

Each experiment runs `N_TIMES` times on the dataset, named with the model so it sits beside the lab's runs in Langfuse. `before_each_run` is for the customer notes in challenge 2.

In [ ]:
def run_experiment(name, description, agent, max_concurrency, runs=N_TIMES, before_each_run=None):
    """Run an agent on the claims `runs` times as Langfuse experiments, and return every run's claims in one table."""

    async def task(*, item, **kwargs):
        return await decide(agent, item.input)

    tables = []
    for run in range(1, runs + 1):
        recorded.clear()
        if before_each_run:
            before_each_run()
        result = langfuse.run_experiment(
            name=f"{name} (Haiku 5.5)",
            description=f"{description}, on {MODEL}",
            data=items,
            task=task,
            evaluators=[score_claim],
            run_evaluators=[score_run],
            max_concurrency=max_concurrency,
            metadata={"model": MODEL},
        )
        print(result.format())
        tables.append(score_table(result.item_results).assign(run=run, experiment=name))
    return pd.concat(tables, ignore_index=True)

### Experiment 1: the payout tool

The lab's first agent, with no memory.

In [ ]:
payout_runs = run_experiment("Payout tool", "The agent with the payout tool", payout_agent, max_concurrency=5)

### Experiment 2: the payout tool and Qdrant

The lab's memory agent, searching by meaning.

In [ ]:
qdrant_runs = run_experiment("Payout tool and memory", "The agent with the payout tool and a memory of the knowledge base", qdrant_agent, max_concurrency=1)

### Experiment 3: the payout tool and the word lookup

In [ ]:
words_runs = run_experiment("Payout tool and word lookup", "The agent with the payout tool and a word lookup on the knowledge base", words_agent, max_concurrency=5)

### Searching by meaning against searching by words

In [ ]:
challenge_1 = {"Payout tool": payout_runs, "Qdrant": qdrant_runs, "Word lookup": words_runs}
pd.DataFrame([average_scores(table) for table in challenge_1.values()], index=list(challenge_1)).round(3)

For each claim that needs a fact, how many runs found it with each search, and how many runs got the claim right.

In [ ]:
def by_claim(tables):
    """For each claim that needs a fact: the runs that found it and the runs that got the claim right, for each experiment."""
    rows = []
    for name, table in tables.items():
        needs = table[table["found_fact"].notna()]
        for claim_id, runs in needs.groupby("claim_id"):
            rows.append({"claim_id": claim_id, "experiment": name, "found": f"{int(runs['found_fact'].sum())}/{len(runs)}", "right": f"{int(runs['right'].sum())}/{len(runs)}"})
    return pd.DataFrame(rows).pivot(index="claim_id", columns="experiment", values=["found", "right"])


by_claim({"Qdrant": qdrant_runs, "Word lookup": words_runs})

In [ ]:
for name, table in challenge_1.items():
    show_confusion_matrix(table, name)

## Challenge 2: a memory about each customer

Each customer has a Markdown file of notes in `customer_notes/`, named by their `customer_id`. The agent reads a customer's notes before it decides, and adds a line about the claim after it records its decision.

Each customer has only one claim in this set, so I wrote notes for six of them, the kind a contact centre keeps. They don't settle any claim: they're there to see the agent read them, and what happens when its own notes come back.

In [ ]:
NOTES = Path("customer_notes")

SEED_NOTES = {
    "CUS-3301": ["2026-02-14: Booked the Bahamas trip through a travel agent, Bluewater Travel, and asked for copies of our letters to go to the agent."],
    "CUS-3307": ["2025-11-04: Called about a bag that arrived a day late in Lisbon. It came the next morning and they didn't claim.", "Prefers email to phone calls."],
    "CUS-3314": ["2026-03-12: Asked before the trip whether diving is covered. Pointed them to the activities section of the policy wording, with no promise either way."],
    "CUS-3316": ["2026-04-30: Asked for the policy documents again, by email, before the Costa Rica trip."],
    "CUS-3320": ["2026-01-20: New address in Guadalajara. Writes in Spanish or English, and replies in English are fine."],
    "CUS-3323": ["2026-05-08: Asked whether the policy pays for an emergency passport. Sent them the wording."],
}


def notes_path(customer_id):
    """The file of a customer's notes, for a customer id like CUS-3301."""
    if not re.fullmatch(r"CUS-\d+", customer_id):
        raise ValueError(f"{customer_id} is not a customer id")
    return NOTES / f"{customer_id}.md"


def reset_notes():
    """Put the customer notes back to the ones I wrote."""
    shutil.rmtree(NOTES, ignore_errors=True)
    NOTES.mkdir()
    for customer_id, notes in SEED_NOTES.items():
        notes_path(customer_id).write_text(f"# Notes on {customer_id}\n\n" + "".join(f"- {note}\n" for note in notes), encoding="utf-8")


@tool(parse_docstring=True)
def read_customer_notes(customer_id: str) -> str:
    """Read UnderCover's notes on a customer: earlier calls, preferences and claims.

    Args:
        customer_id: The customer's id, such as CUS-3301
    """
    path = notes_path(customer_id)
    return path.read_text(encoding="utf-8") if path.exists() else f"There are no notes on {customer_id}"


@tool(parse_docstring=True)
def add_customer_note(customer_id: str, note: str) -> str:
    """Add a note to a customer's notes, for whoever deals with them next.

    Args:
        customer_id: The customer's id, such as CUS-3301
        note: One line to add
    """
    path = notes_path(customer_id)
    if not path.exists():
        path.write_text(f"# Notes on {customer_id}\n\n", encoding="utf-8")
    with path.open("a", encoding="utf-8") as notes:
        notes.write(f"- {note.strip()}\n")
    return f"Added the note to {customer_id}'s notes"


reset_notes()
print(read_customer_notes.invoke({"customer_id": "CUS-3307"}))

The agent from Experiment 2, with the notes tools and one more line in its instructions.

In [ ]:
NOTES_LINE = "Before you decide, read the customer's notes. After you record your decision, add a one-line note about this claim to the customer's notes.\n"

notes_agent = create_agent(
    model=haiku,
    tools=[record_decision, calculate_payout, qdrant_find, read_customer_notes, add_customer_note],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{MEMORY_LINE}{NOTES_LINE}\n{policy_wording}\n",
    middleware=[ToolRetryMiddleware(max_retries=3)],
)

### What should happen to the notes between runs?

The runs of an experiment are meant to be repeats of the same test. If the notes carry over, the second run starts with a note about every claim from the first run, including the decision it made, so it's no longer the same test: the agent can copy its earlier answer, right or wrong, or take its own note for an earlier claim. So I put the notes back to the ones I wrote before each run (Experiment 4).

To see what carrying them over does, Experiment 4b runs once more without putting them back, starting from the notes the last run of Experiment 4 left.

### Experiment 4: Qdrant and customer notes, put back before each run

In [ ]:
notes_runs = run_experiment(
    "Payout tool, memory and customer notes",
    "The agent with the payout tool, a memory of the knowledge base and customer notes, put back before each run",
    notes_agent,
    max_concurrency=1,
    before_each_run=reset_notes,
)

The notes the last run left, on a customer I wrote notes for and one I didn't.

In [ ]:
for customer_id in ["CUS-3307", "CUS-3321"]:
    print(read_customer_notes.invoke({"customer_id": customer_id}))
added = sum(path.read_text(encoding="utf-8").count("\n- ") for path in NOTES.glob("*.md")) - sum(len(notes) for notes in SEED_NOTES.values())
print(f"The last run added {added} notes across {len(list(NOTES.glob('*.md')))} customers")

### Experiment 4b: the same agent, with the notes kept from the last run

In [ ]:
kept_runs = run_experiment(
    "Payout tool, memory and customer notes kept",
    "The agent with the payout tool, a memory of the knowledge base and the customer notes kept from the last run",
    notes_agent,
    max_concurrency=1,
    runs=1,
)

The claims where keeping the notes changed the decision or the payout, against the run that wrote them, and how many of the kept run's rationales mention an earlier note or decision.

In [ ]:
last_run = notes_runs[notes_runs["run"] == notes_runs["run"].max()].set_index("claim_id")
kept = kept_runs.set_index("claim_id")
changed = kept[(kept["agent"] != last_run["agent"]) | ((kept["agent_payout"] - last_run["agent_payout"]).abs() > 1)]
display(pd.DataFrame({
    "handler": changed["handler"],
    "notes put back": last_run.loc[changed.index, "agent"] + " $" + last_run.loc[changed.index, "agent_payout"].round(2).astype(str),
    "notes kept": changed["agent"] + " $" + changed["agent_payout"].round(2).astype(str),
    "rationale with the notes kept": changed["rationale"],
}))

mentions = kept["rationale"].str.contains(r"note|previous|earlier|already|duplicate", case=False)
print(f"{len(changed)} of {len(kept)} claims changed, and {mentions.sum()} rationales mention an earlier note or decision")

## Stretch: a research sub-agent

The retrieval moves into a sub-agent with `qdrant-find` and the two notes tools. The UnderCover agent keeps `record_decision` and the payout tool, and gets one more: `research`, which sends the sub-agent a request and returns its answer. The sub-agent starts a new conversation for each request, so it knows only what the request tells it.

The sub-agent runs inside the tool, so its model calls show up in Langfuse nested under the `research` call, and it hands its messages to `decide` through `SUB_AGENT_MESSAGES`, the ones before an error too.

In [ ]:
RESEARCH_INSTRUCTIONS = """You look things up for UnderCover's claims handlers.
When you are asked about a product, an event or a place, search the knowledge base, and answer with the facts you found, word for word. Say when you found nothing.
When you are asked about a customer, read their notes and answer with them. When you are asked to add a note to a customer's notes, add it.
"""

researcher = create_agent(
    model=haiku,
    tools=[qdrant_find, read_customer_notes, add_customer_note],
    system_prompt=RESEARCH_INSTRUCTIONS,
    middleware=[ToolRetryMiddleware(max_retries=3)],
)


@tool(parse_docstring=True)
async def research(request: str) -> str:
    """Ask the research assistant to look up products, events and places in the knowledge base, to read a customer's notes, or to add a note to them.

    Args:
        request: What to look up or note, with the names, places and dates from the claim and the customer's id
    """
    messages = []
    try:
        async for state in researcher.astream({"messages": [{"role": "user", "content": request}]}, stream_mode="values"):
            messages = state["messages"]
    finally:
        SUB_AGENT_MESSAGES.get().extend(messages)
    return messages[-1].text


RESEARCH_LINE = "Before you decide, ask the research assistant about any product, event or place in the claim that you need to know more about, and for the customer's notes. After you record your decision, ask it to add a one-line note about this claim to the customer's notes.\n"

research_agent = create_agent(
    model=haiku,
    tools=[record_decision, calculate_payout, research],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{RESEARCH_LINE}\n{policy_wording}\n",
)

### Experiment 5: the research sub-agent, with the notes put back before each run

In [ ]:
research_runs = run_experiment(
    "Payout tool and research sub-agent",
    "The agent with the payout tool and a research sub-agent with the memory and customer notes, put back before each run",
    research_agent,
    max_concurrency=1,
    before_each_run=reset_notes,
)

## All the experiments

In [ ]:
experiments = {
    "1. Payout tool": payout_runs,
    "2. Qdrant": qdrant_runs,
    "3. Word lookup": words_runs,
    "4. Qdrant and notes, put back": notes_runs,
    "4b. Qdrant and notes, kept": kept_runs,
    "5. Research sub-agent": research_runs,
}
pd.DataFrame([average_scores(table) for table in experiments.values()], index=list(experiments)).round(3)

In [ ]:
by_claim({"Qdrant": qdrant_runs, "Word lookup": words_runs, "Sub-agent": research_runs})

In [ ]:
for name in ["4. Qdrant and notes, put back", "5. Research sub-agent"]:
    show_confusion_matrix(experiments[name], name)

The claims each experiment got wrong in every run, with the last run's rationale.

In [ ]:
always_wrong = []
for name, table in experiments.items():
    wrong = table.groupby("claim_id").filter(lambda runs: not runs["right"].any())
    last = wrong.sort_values("run").groupby("claim_id").tail(1)
    always_wrong.append(last.assign(experiment=name)[["experiment", "claim_id", "handler", "agent", "handler_payout", "agent_payout", "found_fact", "rationale"]])
pd.concat(always_wrong, ignore_index=True)

## The results from Langfuse

The same comparison read back from Langfuse, so it works without the runs in this kernel. Each row averages an experiment's runs on the whole dataset; runs on part of it, like a `TINY` check or the lab's free version, are left out.

In [ ]:
SUMMARY = ["runs", "accuracy", "decision_accuracy", "kappa", "cost_of_mistakes", "straight_through", "found_fact", "searches_a_claim", "llm_cost", "seconds_a_claim"]


def langfuse_runs(dataset):
    """Every scored run on the whole dataset in Langfuse, one row each, with its run scores."""
    runs, cursor = [], None
    while True:
        page = langfuse.api.experiments.list(from_start_time=dataset.created_at, dataset_id=dataset.id, fields="core,scores", limit=100, cursor=cursor)
        runs += page.data
        cursor = page.meta.cursor
        if not cursor:
            break
    return pd.DataFrame([
        {"experiment": run.name.rsplit(" - ", 1)[0], "started": run.start_time, **{score.name: score.value for score in run.scores or []}}
        for run in runs
        if run.item_count == len(dataset.items) and run.scores
    ])


# Langfuse sends the scores in the background, so send what is left before reading them back
langfuse.flush()
runs = langfuse_runs(langfuse.get_dataset(DATASET))
if runs.empty:
    print("No runs on the whole dataset yet")
else:
    runs = runs.sort_values("started")
    summary = runs.groupby("experiment", sort=False).mean(numeric_only=True).assign(runs=runs.groupby("experiment", sort=False).size())
    display(summary[[column for column in SUMMARY if column in summary]].round(3))

## What I found

Two runs of each experiment on the 25 even harder claims, on Haiku 5.5 (one run for 4b). The 11 runs cost $0.91 in model calls. A difference of one claim in one run moves the accuracy by 0.04, so two runs show which differences repeat, not how big they'd be over many runs.

| | 1. Payout tool | 2. Qdrant | 3. Word lookup | 4. Qdrant and notes | 4b. Notes kept | 5. Sub-agent |
|---|---|---|---|---|---|---|
| Accuracy (each run) | 0.38 (0.32, 0.44) | 0.96 (0.96, 0.96) | **1.00** (1.00, 1.00) | 0.96 (0.96, 0.96) | 0.84 | 0.96 (0.92, 1.00) |
| Cost of mistakes, a run | $1,571.95 | $39.20 | **$0** | $39.20 | $178.40 | $775.00 |
| found_fact | 0.00 | 1.00 | 1.00 | 1.00 | 1.00 | 1.00 |
| Searches a claim | 0 | 1.98 | 1.78 | 1.22 | 1.24 | 1.44 |
| Model cost, a run | $0.055 | $0.088 | $0.076 | $0.088 | $0.089 | $0.103 |
| Seconds a claim | 8.6 | 16.7 | 9.4 | 13.8 | 14.3 | 22.1 |

### Without a memory, the agent refers what it can't check

With only the payout tool, the agent got 0.38 right. It mostly referred: it can't tell when a storm was named or a strike announced (the section 9 exclusion), how deep a dive went or what grade a rapid is (section 8), or whether a Halden Arc is electronics, and it said so in its rationales. Where it didn't refer, it guessed: it approved CLM-1303 and CLM-1305, which the handler declined because the strike and the storm were public before the policy was bought, and it overpaid CLM-1310 and CLM-1312 by counting a smart ring and a handheld GPS as electronics. Every memory below found the fact for all 19 claims that need one, in every run.

### Challenge 1: the word lookup did as well as searching by meaning, for less

The word lookup got every claim right in both runs. Qdrant got 0.96 in both, and the claim it got wrong, CLM-1323, needs no fact: it referred it once and paid $177 instead of $205.40 once, leaving out a taxi receipt that covers the police station and the embassy together. That's a judgment on the wording, not a retrieval miss, and the word lookup paid it in full both times, so I don't put the difference down to the search.

Searching by words works here because every claim names what it turns on: the storm, the strike's carrier or city, the product, the dive site, the river section, the operator, and the facts start with those names. The paraphrase in the lab shows where it breaks: "a gadget for watching films on a wall" finds the Sentinel Wall dive site by its word "wall", and Qdrant finds the projector. A customer who writes "my little projector" without the brand would need Qdrant.

The word lookup is also cheaper and faster: 14% less a run and 9.4 seconds a claim against 16.7. Each Qdrant search starts the MCP server and loads its embedding model, and the local database takes one process at a time, so its runs had to go one claim at a time (7 minutes a run), where the word lookup ran five at once (1 minute).

### Challenge 2: put the notes back before each run

With the notes put back before each run, the scores were Qdrant's: 0.96 in both runs, with one claim wrong in each (CLM-1319 referred over whether a jet-boat ride is rafting, then CLM-1323's taxi again). The notes I wrote don't bear on any decision, so that's what I expected. In the last run the agent added a line to each of the 25 customers' notes, such as *"CLM-1307: approved for $562.95. The Halden Arc projector was capped at the $300 per-item limit because it is not a listed electronic."* It also searched less (1.2 searches a claim against 2.0), with no loss in the facts it found.

With the notes kept from the last run, accuracy fell to 0.84. The agent referred three claims it had approved before (CLM-1315, CLM-1319, CLM-1322), each time because of its own note: *"the customer notes say CLM-1319 was already approved for $432.40, so approving it again could pay twice."* That's a careful handler's answer to a note saying the claim was already paid, but the run is no longer a repeat of the first: it tests a different situation, with different right answers.

So the notes go back to the same starting point before every run, as the memory does: each run is then the same test. Kept notes would be worth testing on purpose, with a second claim from the same customer in the dataset and notes written for it, not left over from an earlier run.

### Stretch: the sub-agent found every fact and cost the most

The research sub-agent found the fact every time and its second run got every claim right. Its first run made two mistakes, neither about retrieval: it declined CLM-1304, reading a 30-hour delay as 6.25 hours by putting the scheduled 08:15 departure on the same day as the actual 14:30 one ($1,500 as a wrongful decline), and it referred CLM-1321 because the payout tool asked for sign-off. It's the slowest and dearest of the six (22 seconds and $0.103 a run), since every question is a second conversation with its own model calls. With one or two short facts a claim there's nothing for it to keep out of the UnderCover agent's context; it would earn its cost with a larger knowledge base, or long customer histories to summarise.

### What I'd choose

For this knowledge base, the word lookup: it found every fact, made no mistakes in either run, and is the cheapest and fastest of the memories. I'd keep Qdrant for the day claims describe things instead of naming them, and I'd test that first, with claims written that way. For the customer notes, the setting matters more than the tool: each run starts from the same notes, or the scores measure the agent's own earlier answers.